# Week 6：LOF 建模（Gavin / Jiahao）

本 notebook 完成 Gavin 在 Week 6 的任务：基于 Lee 输出的共享 ML-ready dataset 建立 Local Outlier Factor（LOF）模型，测试多个 `n_neighbors` 参数，并导出 LOF anomaly score 和 anomaly label。

本 notebook 只使用 Lee 在 `feature_list.json` 中列出的 `model_feature_columns`。因此，`timestamp`、`zone` 等 identifier，以及 v3 statistical baseline label，都不会进入 LOF 模型训练。

## Cell 1：配置和导入库

这个 cell 设置文件路径、模型参数，并导入 notebook 会使用的 Python 库。`SAMPLE_SIZE = None` 表示 LOF 使用 Lee 的完整数据集，而不是抽样数据。本 notebook 测试一个 `3 x 3` 参数组合：3 个 `n_neighbors` 和 3 个 `contamination`。

In [1]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.neighbors import LocalOutlierFactor

os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 1))

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "ml" else Path.cwd()
ML_READY_DIR = PROJECT_DIR / "outputs" / "floor6_zone1245_ml_ready"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "floor6_zone1245_lof_modelling_gavin"

ML_READY_PATH = ML_READY_DIR / "floor6_zone1245_ml_ready_2018_jul01_sep01.csv.gz"
FEATURE_LIST_PATH = ML_READY_DIR / "feature_list.json"

RANDOM_STATE = 42
SAMPLE_SIZE = None
NEIGHBOR_SETTINGS = [20, 50, 100]
CONTAMINATION_SETTINGS = [0.03, 0.05, 0.09]
SELECTED_N_NEIGHBORS = 100
SELECTED_CONTAMINATION = 0.05
N_JOBS = -1

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"项目目录: {PROJECT_DIR}")
print(f"ML-ready 数据集: {ML_READY_PATH}")
print(f"输出目录: {OUTPUT_DIR}")

项目目录: d:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis
ML-ready 数据集: d:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\floor6_zone1245_ml_ready\floor6_zone1245_ml_ready_2018_jul01_sep01.csv.gz
输出目录: d:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\floor6_zone1245_lof_modelling_gavin


## Cell 2：读取 Lee 的 ML-ready 数据

这个 cell 读取 Lee 的共享数据集和 feature manifest。它会检查所有模型特征是否存在、是否都是数值型，并确认后续比较需要的参考字段是否保留在数据中。

In [2]:

if not ML_READY_PATH.exists():
    raise FileNotFoundError(
        f"找不到 {ML_READY_PATH}。请先运行 Lee 的 ML_Data_Preparation_Feature_Engineering.ipynb。"
    )

feature_manifest = json.loads(FEATURE_LIST_PATH.read_text(encoding="utf-8"))
model_features = feature_manifest["model_feature_columns"]
baseline_reference_columns = feature_manifest.get(
    "baseline_reference_columns",
    ["baseline_anomaly_label", "baseline_anomaly_count"],
)
baseline_method_label_columns = feature_manifest.get("baseline_method_label_columns", {})
baseline_method_pairs = [
    (method, column)
    for method, column in baseline_method_label_columns.items()
    if isinstance(column, str)
]

base_reference_columns = ["timestamp", "floor", "zone", "hour", "operating_period"]
reference_columns = base_reference_columns + [
    column for column in baseline_reference_columns if column not in base_reference_columns
]

ml_ready = pd.read_csv(ML_READY_PATH, parse_dates=["timestamp"])

missing_features = sorted(set(model_features) - set(ml_ready.columns))
missing_reference = sorted(set(reference_columns) - set(ml_ready.columns))
if missing_features or missing_reference:
    raise ValueError(
        f"缺少模型特征: {missing_features}; 缺少参考字段: {missing_reference}"
    )

baseline_method_pairs = [
    (method, column) for method, column in baseline_method_pairs if column in ml_ready.columns
]

non_numeric_features = ml_ready[model_features].select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric_features:
    raise TypeError(f"LOF 需要数值型特征。非数值型字段: {non_numeric_features}")

if ml_ready[model_features].isna().any().any():
    raise ValueError("模型特征中存在 NaN。请重新检查 Lee 的 preprocessing 输出。")

if not np.isfinite(ml_ready[model_features].to_numpy(dtype=np.float32)).all():
    raise ValueError("模型特征中存在 infinite value。请重新检查 Lee 的 preprocessing 输出。")

baseline_rate = float(ml_ready["baseline_anomaly_label"].mean())

print(f"读取行数: {len(ml_ready):,}")
print(f"模型特征数量: {len(model_features)}")
print(f"用于比较的 v3 baseline 异常比例: {baseline_rate:.4%}")
print(f"baseline 方法标签: {[method for method, _ in baseline_method_pairs]}")
print(f"LOF contamination 参数: {CONTAMINATION_SETTINGS}")
ml_ready[reference_columns].head()


读取行数: 362,880
模型特征数量: 47
用于比较的 v3 baseline 异常比例: 12.8580%
baseline 方法标签: ['high_quantile_threshold', 'low_quantile_threshold', 'operational_schedule_mismatch', 'robust_z_score', 'zero_aware_threshold']
LOF contamination 参数: [0.03, 0.05, 0.09]


,timestamp,floor,zone,hour,operating_period,baseline_anomaly_label,baseline_anomaly_count,baseline_anomaly_methods,baseline_anomaly_variables,baseline_anomaly_directions,baseline_robust_z_score_count,baseline_high_quantile_count,baseline_low_quantile_count,baseline_zero_aware_count,baseline_operational_count,baseline_robust_z_score_label,baseline_high_quantile_label,baseline_low_quantile_label,baseline_zero_aware_label,baseline_operational_label
0,2018-07-01 00:00:00,6,1,0,weekend,0,0,none,none,none,0,0,0,0,0,0,0,0,0,0
1,2018-07-01 00:01:00,6,1,0,weekend,0,0,none,none,none,0,0,0,0,0,0,0,0,0,0
2,2018-07-01 00:02:00,6,1,0,weekend,0,0,none,none,none,0,0,0,0,0,0,0,0,0,0
3,2018-07-01 00:03:00,6,1,0,weekend,0,0,none,none,none,0,0,0,0,0,0,0,0,0,0
4,2018-07-01 00:04:00,6,1,0,weekend,0,0,none,none,none,0,0,0,0,0,0,0,0,0,0


## Cell 3：准备 LOF 建模数据

这个 cell 准备 LOF 使用的数据。因为 `SAMPLE_SIZE = None`，所以 notebook 会使用 Lee 的完整 ML-ready 数据。这里仍然保留分层抽样函数，方便之后如果运行时间太长，可以快速改成抽样实验。

In [3]:
def make_stratified_sample(frame: pd.DataFrame, sample_size: int | None) -> pd.DataFrame:
    if sample_size is None or sample_size >= len(frame):
        return frame.copy()

    strata = (
        frame["zone"].astype(str)
        + "_"
        + frame["operating_period"].astype(str)
        + "_"
        + frame["baseline_anomaly_label"].astype(str)
    )

    if strata.value_counts().min() < 2:
        return frame.sample(n=sample_size, random_state=RANDOM_STATE).copy()

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        train_size=sample_size,
        random_state=RANDOM_STATE,
    )
    sample_index, _ = next(splitter.split(frame, strata))
    return frame.iloc[sample_index].sort_values("timestamp").copy()


lof_data = make_stratified_sample(ml_ready, SAMPLE_SIZE)
X = lof_data[model_features].to_numpy(dtype=np.float32)

sample_check = pd.DataFrame(
    {
        "dataset": ["Lee 完整数据集", "LOF 建模数据"],
        "rows": [len(ml_ready), len(lof_data)],
        "baseline_anomaly_rate": [ml_ready["baseline_anomaly_label"].mean(), lof_data["baseline_anomaly_label"].mean()],
        "zones": [ml_ready["zone"].nunique(), lof_data["zone"].nunique()],
        "operating_periods": [ml_ready["operating_period"].nunique(), lof_data["operating_period"].nunique()],
    }
)
sample_check

,dataset,rows,baseline_anomaly_rate,zones,operating_periods
0,Lee 完整数据集,362880,0.12858,4,6
1,LOF 建模数据,362880,0.12858,4,6


## Cell 4：训练 LOF 并测试 `n_neighbors` 与 `contamination`

这个 cell 会对每个 `n_neighbors` 拟合一次 LOF，然后在同一组 LOF score 排名上应用 3 个 `contamination` 阈值。这样既保留 `3 x 3` 参数比较，又避免对同一个 `n_neighbors` 重复计算邻域密度。这里把 LOF score 转换成“数值越高越异常”的方向。v3 baseline label 只用于结果比较，不作为模型特征。

In [4]:

def contamination_suffix(contamination: float) -> str:
    return f"c{int(round(contamination * 1000)):03d}"


def labels_from_score(score: np.ndarray, contamination: float) -> np.ndarray:
    anomaly_count = int(round(len(score) * contamination))
    if anomaly_count <= 0:
        return np.zeros(len(score), dtype=int)
    ranked_index = np.argsort(score)[::-1]
    labels = np.zeros(len(score), dtype=int)
    labels[ranked_index[:anomaly_count]] = 1
    return labels


results = lof_data[reference_columns].copy()
parameter_rows = []

for n_neighbors in NEIGHBOR_SETTINGS:
    if n_neighbors >= len(lof_data):
        raise ValueError(f"n_neighbors={n_neighbors} 必须小于建模数据行数。")

    start_time = time.time()
    model = LocalOutlierFactor(
        n_neighbors=n_neighbors,
        contamination="auto",
        n_jobs=N_JOBS,
    )
    model.fit_predict(X)
    fit_seconds = time.time() - start_time

    score_column = f"lof_score_k{n_neighbors}"
    results[score_column] = -model.negative_outlier_factor_

    for contamination in CONTAMINATION_SETTINGS:
        suffix = f"k{n_neighbors}_{contamination_suffix(contamination)}"
        label_column = f"lof_label_{suffix}"
        results[label_column] = labels_from_score(results[score_column].to_numpy(), contamination)

        lof_mask = results[label_column].eq(1)
        baseline_mask = results["baseline_anomaly_label"].eq(1)
        lof_count = int(lof_mask.sum())
        baseline_count = int(baseline_mask.sum())
        overlap_count = int((lof_mask & baseline_mask).sum())
        union_count = int((lof_mask | baseline_mask).sum())
        agreement_count = int((results[label_column] == results["baseline_anomaly_label"]).sum())

        parameter_row = {
            "n_neighbors": n_neighbors,
            "contamination": contamination,
            "rows_modelled": len(results),
            "lof_anomaly_count": lof_count,
            "lof_anomaly_rate": lof_count / len(results),
            "baseline_anomaly_count": baseline_count,
            "baseline_anomaly_rate": baseline_rate,
            "baseline_overlap_count": overlap_count,
            "baseline_overlap_share_of_lof": overlap_count / lof_count if lof_count else np.nan,
            "jaccard_with_baseline": overlap_count / union_count if union_count else np.nan,
            "method_agreement_rate": agreement_count / len(results),
            "label_column": label_column,
            "score_column": score_column,
            "fit_seconds": fit_seconds,
        }
        for method, method_label_column in baseline_method_pairs:
            method_mask = results[method_label_column].eq(1)
            parameter_row[f"{method}_baseline_count"] = int(method_mask.sum())
            parameter_row[f"{method}_overlap_count"] = int((lof_mask & method_mask).sum())
        parameter_rows.append(parameter_row)

parameter_summary = pd.DataFrame(parameter_rows)
parameter_summary


c:\Users\jh\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] 系统找不到指定的文件。
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\jh\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\jh\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\jh\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\jh\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                      

,n_neighbors,contamination,rows_modelled,lof_anomaly_count,lof_anomaly_rate,baseline_anomaly_count,baseline_anomaly_rate,baseline_overlap_count,baseline_overlap_share_of_lof,jaccard_with_baseline,...,high_quantile_threshold_baseline_count,high_quantile_threshold_overlap_count,low_quantile_threshold_baseline_count,low_quantile_threshold_overlap_count,operational_schedule_mismatch_baseline_count,operational_schedule_mismatch_overlap_count,robust_z_score_baseline_count,robust_z_score_overlap_count,zero_aware_threshold_baseline_count,zero_aware_threshold_overlap_count
0,20,0.03,362880,10886,0.029999,46659,0.12858,1696,0.155796,0.030368,...,4983,184,28929,1188,456,37,20011,653,46,0
1,20,0.05,362880,18144,0.050000,46659,0.12858,2894,0.159502,0.046746,...,4983,347,28929,2019,456,48,20011,1079,46,0
2,20,0.09,362880,32659,0.089999,46659,0.12858,5252,0.160813,0.070910,...,4983,672,28929,3589,456,66,20011,1889,46,2
3,50,0.03,362880,10886,0.029999,46659,0.12858,2052,0.188499,0.036978,...,4983,309,28929,1566,456,51,20011,349,46,0
4,50,0.05,362880,18144,0.050000,46659,0.12858,3262,0.179784,0.053005,...,4983,513,28929,2491,456,52,20011,495,46,4
5,50,0.09,362880,32659,0.089999,46659,0.12858,5515,0.168866,0.074726,...,4983,885,28929,4274,456,57,20011,796,46,9
6,100,0.03,362880,10886,0.029999,46659,0.12858,1895,0.174077,0.034052,...,4983,274,28929,1047,456,0,20011,628,46,0
7,100,0.05,362880,18144,0.050000,46659,0.12858,2874,0.158399,0.046408,...,4983,460,28929,1605,456,0,20011,870,46,0
8,100,0.09,362880,32659,0.089999,46659,0.12858,5206,0.159405,0.070245,...,4983,743,28929,2906,456,3,20011,1684,46,7


## Cell 5：选择主 LOF 结果并保存输出

这个 cell 选择 `n_neighbors = 100` 和 `contamination = 0.05` 作为主 LOF 设置。`k=100` 使用更平滑的邻域定义；`0.05` 比直接使用 v3 baseline 的异常比例更独立。这个 cell 会保存参数比较表、LOF anomaly 结果、top anomaly 预览，以及 JSON 摘要。

In [5]:

selected_suffix = f"k{SELECTED_N_NEIGHBORS}_{contamination_suffix(SELECTED_CONTAMINATION)}"
selected_label = f"lof_label_{selected_suffix}"
selected_score = f"lof_score_k{SELECTED_N_NEIGHBORS}"

results["lof_anomaly_label"] = results[selected_label]
results["lof_anomaly_score"] = results[selected_score]

selected_lof_mask = results["lof_anomaly_label"].eq(1)
baseline_method_overlap = pd.DataFrame(
    [
        {
            "baseline_method": method,
            "baseline_rows": int(results[method_label_column].sum()),
            "lof_overlap_rows": int((selected_lof_mask & results[method_label_column].eq(1)).sum()),
            "lof_overlap_share": (
                int((selected_lof_mask & results[method_label_column].eq(1)).sum())
                / int(selected_lof_mask.sum())
                if int(selected_lof_mask.sum())
                else np.nan
            ),
        }
        for method, method_label_column in baseline_method_pairs
    ]
)

parameter_summary_path = OUTPUT_DIR / "lof_parameter_comparison.csv"
method_overlap_path = OUTPUT_DIR / "lof_baseline_method_overlap.csv"
results_filename = "lof_anomaly_results_full.csv.gz" if SAMPLE_SIZE is None else "lof_anomaly_results_sample.csv.gz"
results_path = OUTPUT_DIR / results_filename
top_anomalies_path = OUTPUT_DIR / "lof_top_anomalies.csv"
summary_path = OUTPUT_DIR / "lof_modelling_summary.json"

parameter_summary.to_csv(parameter_summary_path, index=False)
baseline_method_overlap.to_csv(method_overlap_path, index=False)
results.to_csv(results_path, index=False)
results.sort_values("lof_anomaly_score", ascending=False).head(100).to_csv(top_anomalies_path, index=False)

summary_payload = {
    "input_dataset": str(ML_READY_PATH.relative_to(PROJECT_DIR)),
    "feature_list": str(FEATURE_LIST_PATH.relative_to(PROJECT_DIR)),
    "output_results": str(results_path.relative_to(PROJECT_DIR)),
    "output_parameter_comparison": str(parameter_summary_path.relative_to(PROJECT_DIR)),
    "output_baseline_method_overlap": str(method_overlap_path.relative_to(PROJECT_DIR)),
    "sample_size": None if SAMPLE_SIZE is None else int(SAMPLE_SIZE),
    "rows_available": int(len(ml_ready)),
    "rows_modelled": int(len(results)),
    "model_feature_count": int(len(model_features)),
    "n_neighbors_tested": NEIGHBOR_SETTINGS,
    "contamination_tested": CONTAMINATION_SETTINGS,
    "selected_n_neighbors": SELECTED_N_NEIGHBORS,
    "selected_contamination": SELECTED_CONTAMINATION,
    "baseline_anomaly_rate": baseline_rate,
    "baseline_method_label_columns": dict(baseline_method_pairs),
    "baseline_label_used_as_feature": False,
    "identifier_columns_used_as_features": False,
}
summary_path.write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")

print(f"已保存参数比较表: {parameter_summary_path}")
print(f"已保存 baseline 方法 overlap: {method_overlap_path}")
print(f"已保存 LOF anomaly 结果: {results_path}")
print(f"已保存 top anomaly 预览: {top_anomalies_path}")
print(f"已保存建模摘要: {summary_path}")

parameter_summary


已保存参数比较表: d:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\floor6_zone1245_lof_modelling_gavin\lof_parameter_comparison.csv
已保存 baseline 方法 overlap: d:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\floor6_zone1245_lof_modelling_gavin\lof_baseline_method_overlap.csv
已保存 LOF anomaly 结果: d:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\floor6_zone1245_lof_modelling_gavin\lof_anomaly_results_full.csv.gz
已保存 top anomaly 预览: d:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\floor6_zone1245_lof_modelling_gavin\lof_top_anomalies.csv
已保存建模摘要: d:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\floor6_zone1245_lof_mo

,n_neighbors,contamination,rows_modelled,lof_anomaly_count,lof_anomaly_rate,baseline_anomaly_count,baseline_anomaly_rate,baseline_overlap_count,baseline_overlap_share_of_lof,jaccard_with_baseline,...,high_quantile_threshold_baseline_count,high_quantile_threshold_overlap_count,low_quantile_threshold_baseline_count,low_quantile_threshold_overlap_count,operational_schedule_mismatch_baseline_count,operational_schedule_mismatch_overlap_count,robust_z_score_baseline_count,robust_z_score_overlap_count,zero_aware_threshold_baseline_count,zero_aware_threshold_overlap_count
0,20,0.03,362880,10886,0.029999,46659,0.12858,1696,0.155796,0.030368,...,4983,184,28929,1188,456,37,20011,653,46,0
1,20,0.05,362880,18144,0.050000,46659,0.12858,2894,0.159502,0.046746,...,4983,347,28929,2019,456,48,20011,1079,46,0
2,20,0.09,362880,32659,0.089999,46659,0.12858,5252,0.160813,0.070910,...,4983,672,28929,3589,456,66,20011,1889,46,2
3,50,0.03,362880,10886,0.029999,46659,0.12858,2052,0.188499,0.036978,...,4983,309,28929,1566,456,51,20011,349,46,0
4,50,0.05,362880,18144,0.050000,46659,0.12858,3262,0.179784,0.053005,...,4983,513,28929,2491,456,52,20011,495,46,4
5,50,0.09,362880,32659,0.089999,46659,0.12858,5515,0.168866,0.074726,...,4983,885,28929,4274,456,57,20011,796,46,9
6,100,0.03,362880,10886,0.029999,46659,0.12858,1895,0.174077,0.034052,...,4983,274,28929,1047,456,0,20011,628,46,0
7,100,0.05,362880,18144,0.050000,46659,0.12858,2874,0.158399,0.046408,...,4983,460,28929,1605,456,0,20011,870,46,0
8,100,0.09,362880,32659,0.089999,46659,0.12858,5206,0.159405,0.070245,...,4983,743,28929,2906,456,3,20011,1684,46,7


## Cell 6：按 zone、hour 和 operating period 汇总 LOF 异常

这个 cell 生成 Week 6 后续比较需要的基础分布表。这些表展示主 LOF 模型在不同 zone、hour 和 operating period 中检测到的异常分布。

In [6]:

def summarise_distribution(frame: pd.DataFrame, group_column: str) -> pd.DataFrame:
    aggregations = {
        "rows": ("lof_anomaly_label", "size"),
        "lof_anomalies": ("lof_anomaly_label", "sum"),
        "baseline_anomalies": ("baseline_anomaly_label", "sum"),
        "mean_lof_score": ("lof_anomaly_score", "mean"),
    }
    for method, method_label_column in baseline_method_pairs:
        if method_label_column in frame.columns:
            aggregations[f"{method}_baseline_rows"] = (method_label_column, "sum")

    summary = (
        frame.groupby(group_column, dropna=False)
        .agg(**aggregations)
        .reset_index()
    )
    summary["lof_anomaly_rate"] = summary["lof_anomalies"] / summary["rows"]
    return summary.sort_values("lof_anomalies", ascending=False)


zone_summary = summarise_distribution(results, "zone")
hour_summary = summarise_distribution(results, "hour")
period_summary = summarise_distribution(results, "operating_period")

zone_summary.to_csv(OUTPUT_DIR / "lof_summary_by_zone.csv", index=False)
hour_summary.to_csv(OUTPUT_DIR / "lof_summary_by_hour.csv", index=False)
period_summary.to_csv(OUTPUT_DIR / "lof_summary_by_operating_period.csv", index=False)

display(zone_summary)
display(period_summary)
display(hour_summary.head(10))


,zone,rows,lof_anomalies,baseline_anomalies,mean_lof_score,high_quantile_threshold_baseline_rows,low_quantile_threshold_baseline_rows,operational_schedule_mismatch_baseline_rows,robust_z_score_baseline_rows,zero_aware_threshold_baseline_rows,lof_anomaly_rate
3,5,90720,5523,12295,1.096745,1064,5628,0,7535,11,0.060880
1,2,90720,4629,10149,1.080112,1662,6457,456,4045,0,0.051025
0,1,90720,4451,13021,1.081971,1384,9465,0,3607,0,0.049063
2,4,90720,3541,11194,1.071975,873,7379,0,4824,35,0.039032


,operating_period,rows,lof_anomalies,baseline_anomalies,mean_lof_score,high_quantile_threshold_baseline_rows,low_quantile_threshold_baseline_rows,operational_schedule_mismatch_baseline_rows,robust_z_score_baseline_rows,zero_aware_threshold_baseline_rows,lof_anomaly_rate
5,weekend,103680,6834,3273,1.093367,0,0,456,3269,0,0.065914
0,core_work,97200,4714,26866,1.083232,4123,22371,0,6684,0,0.048498
1,evening_off,54000,3014,2760,1.086168,0,0,0,2753,7,0.055815
2,night_off,75600,1969,5882,1.062179,0,0,0,5875,7,0.026045
3,ramp_down,21600,907,5579,1.083167,629,4550,0,1325,32,0.041991
4,ramp_up,10800,706,2299,1.100913,231,2008,0,105,0,0.065370


,hour,rows,lof_anomalies,baseline_anomalies,mean_lof_score,high_quantile_threshold_baseline_rows,low_quantile_threshold_baseline_rows,operational_schedule_mismatch_baseline_rows,robust_z_score_baseline_rows,zero_aware_threshold_baseline_rows,lof_anomaly_rate
11,11,15120,1074,3259,1.102967,491,2475,60,1128,0,0.071032
19,19,15120,1056,697,1.094607,0,0,0,690,7,0.069841
9,9,15120,1036,3183,1.096225,504,2494,11,923,0,0.068519
12,12,15120,1032,3209,1.101643,323,2661,60,885,0,0.068254
6,6,15120,980,697,1.094108,0,0,0,690,7,0.064815
20,20,15120,972,789,1.096902,0,0,0,789,0,0.064286
7,7,15120,957,2299,1.101797,231,2008,0,105,0,0.063294
10,10,15120,925,3235,1.086604,546,2534,60,907,0,0.061177
15,15,15120,864,3327,1.081060,420,2592,60,975,0,0.057143
8,8,15120,824,2468,1.086798,417,2034,0,525,0,0.054497


## Cell 7：Gavin 交接说明

最后这个 cell 会打印给团队的简短交接说明。关键结果文件保存在 `outputs/floor6_zone1245_lof_modelling_gavin/`。对于 Jiahao/Gavin 的 Week 6 交付，最重要的文件是 `lof_parameter_comparison.csv` 和 `lof_anomaly_results_full.csv.gz`。

In [7]:

selected_row = parameter_summary.loc[
    (parameter_summary["n_neighbors"] == SELECTED_N_NEIGHBORS)
    & (parameter_summary["contamination"] == SELECTED_CONTAMINATION)
].iloc[0]

print("Gavin LOF 建模交接说明")
print(f"- Lee ML-ready 可用行数: {len(ml_ready):,}")
print(f"- 本 notebook 实际建模行数: {len(results):,}")
print(f"- 使用特征数量: {len(model_features)} 个 model_feature_columns")
print(f"- 测试的 n_neighbors: {NEIGHBOR_SETTINGS}")
print(f"- 测试的 contamination: {CONTAMINATION_SETTINGS}")
print(f"- 选定的 n_neighbors: {SELECTED_N_NEIGHBORS}")
print(f"- 选定的 contamination: {SELECTED_CONTAMINATION:.0%}")
print(f"- 选定参数下的 LOF anomalies: {int(selected_row['lof_anomaly_count']):,}")
print(f"- 与 v3 baseline 的 overlap: {int(selected_row['baseline_overlap_count']):,}")
if not baseline_method_overlap.empty:
    print("- 与 v3 baseline 各方法 overlap:")
    display(baseline_method_overlap)
print(f"- 结果文件夹: {OUTPUT_DIR}")


Gavin LOF 建模交接说明
- Lee ML-ready 可用行数: 362,880
- 本 notebook 实际建模行数: 362,880
- 使用特征数量: 47 个 model_feature_columns
- 测试的 n_neighbors: [20, 50, 100]
- 测试的 contamination: [0.03, 0.05, 0.09]
- 选定的 n_neighbors: 100
- 选定的 contamination: 5%
- 选定参数下的 LOF anomalies: 18,144
- 与 v3 baseline 的 overlap: 2,874
- 与 v3 baseline 各方法 overlap:


,baseline_method,baseline_rows,lof_overlap_rows,lof_overlap_share
0,high_quantile_threshold,4983,460,0.025353
1,low_quantile_threshold,28929,1605,0.088459
2,operational_schedule_mismatch,456,0,0.000000
3,robust_z_score,20011,870,0.047950
4,zero_aware_threshold,46,0,0.000000


- 结果文件夹: d:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\floor6_zone1245_lof_modelling_gavin
